In [ ]:
import os
import sys

import time
import random

import numpy as np
import matplotlib.pyplot as plt
import torch
import yaml

import graphlearning as gl
import sklearn.datasets as skdatasets

import models.models as Models
import losses.losses as Losses
from models.model_utils import project_l2
from datasets.dataset_utils import graphlearning_to_pyg

from tqdm.auto import tqdm

In [ ]:
seed = 0
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

device = "cpu"

# Model config/checkpoint
exp_file = "configs/single-runs/example-encode-process-decode.yaml"
model_pth = ""  # set to checkpoint .pt

# PDHG solver params
lam = None     # if None, use model_config.cfg.lam
tau = 0.35
sigma = 0.35
iters = 250
objective_rel_tol = 0.01

# 2-moons trial params
n_trials = 30
n_samples = 100
noise = 0.15
k_neighbors = 10
kernel = "gaussian"

# plotting
error_every = 10  # show error bars every N iterations

In [ ]:
def load_model(model_cfg, model_pth, device="cpu"):
    model_class = getattr(Models, model_cfg["model"])
    model = model_class(**model_cfg["cfg"]).float().to(device)

    ckpt = torch.load(model_pth, map_location=device)
    if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
        state = ckpt["model_state_dict"]
    else:
        state = ckpt

    model.load_state_dict(state)
    model.eval()
    return model


def make_two_moons_graph(n_samples, noise, k_neighbors, kernel, random_state):
    X, _ = skdatasets.make_moons(n_samples=n_samples, noise=noise, random_state=random_state)
    W = gl.weightmatrix.knn(X, k=k_neighbors, kernel=kernel)
    W.setdiag(0)
    W.eliminate_zeros()
    return graphlearning_to_pyg(X, W)


@torch.no_grad()
def pdhg_net_init(model, data, device="cpu"):
    data = data.to(device)
    src, dst = data.edge_index
    e0 = data.x[src] - data.x[dst]
    U0, P0 = model(
        h=data.x.float(),
        e=e0.float(),
        edge_index=data.edge_index.long(),
        w=data.edge_attr.float(),
        x=data.x.float(),
    )
    return U0.detach(), P0.detach()


def random_init(X, m, lam, w, seed=None):
    n, d = X.shape
    g = torch.Generator(device=X.device)
    if seed is not None:
        g.manual_seed(seed)

    mean = X.mean(dim=0, keepdim=True)
    std = X.std(dim=0, keepdim=True, unbiased=False).clamp_min(1e-6)
    U0 = mean + std * torch.randn((n, d), generator=g, device=X.device, dtype=X.dtype)

    sqrtw = w.sqrt()
    r = lam * sqrtw
    Praw = torch.randn((m, d), generator=g, device=X.device, dtype=X.dtype)
    P0 = project_l2(Praw, r)
    return U0, P0


def eval_metrics(U, P, X, src, dst, w, lam):
    obj = float(Losses.energy(U, X, src, dst, w, lam).item())
    pdg = float(Losses.energy_pdg(U, X, P, src, dst, w, lam).item())
    return obj, pdg


def run_vanilla_pdhg(X, src, dst, w, lam, tau, sigma, iters, U0, P0):
    sqrtw = w.sqrt()
    n, _ = X.shape

    U = U0.clone()
    P = P0.clone()

    hist = {"obj": [], "pdg": []}
    obj, pdg = eval_metrics(U, P, X, src, dst, w, lam)
    hist["obj"].append(obj)
    hist["pdg"].append(pdg)

    r = lam * sqrtw
    for _ in range(iters):
        diff = U[src] - U[dst]
        P = project_l2(P + tau * (sqrtw[:, None] * diff), r)
        U = (U + sigma * (X - Losses.divergence(sqrtw[:, None] * P, src, dst, n))) / (1.0 + sigma)

        obj, pdg = eval_metrics(U, P, X, src, dst, w, lam)
        hist["obj"].append(obj)
        hist["pdg"].append(pdg)

    return U.detach(), P.detach(), hist


In [ ]:
# Load model config
with open(exp_file, "r") as f:
    exp_cfg = yaml.safe_load(f)

model_cfg = exp_cfg["model_config"]
if lam is None:
    lam = float(model_cfg["cfg"].get("lam", 1.0))

if not model_pth:
    raise ValueError("Set model_pth to a trained PDHG-Net checkpoint.")

model = load_model(model_cfg, model_pth, device=device)
print(f"Loaded model: {model_cfg['model']}")
print(f"lam={lam}, tau={tau}, sigma={sigma}, iters={iters}")

In [ ]:
methods = ["data", "random", "pdhg_net"]
results = {
    m: {
        "obj": [],
        "pdg": [],
        "init_time_s": [],
        "solve_time_s": [],
        "total_time_s": [],
    }
    for m in methods
}

trials = []
for t in range(n_trials):
    g = make_two_moons_graph(
        n_samples=n_samples,
        noise=noise,
        k_neighbors=k_neighbors,
        kernel=kernel,
        random_state=seed + t,
    )
    trials.append(g)

for t, g in enumerate(tqdm(trials, desc="Trials")):
    data = g.to(device)
    X = data.x.float()
    src, dst = data.edge_index.long()
    w = data.edge_attr.float()
    m = src.numel()
    d = X.shape[1]

    # data init
    t0 = time.perf_counter()
    U0_data, P0_data = X.clone(), torch.zeros((m, d), device=device, dtype=X.dtype)
    t_init = time.perf_counter() - t0

    t0 = time.perf_counter()
    _, _, hist_data = run_vanilla_pdhg(X, src, dst, w, lam, tau, sigma, iters, U0_data, P0_data)
    t_solve = time.perf_counter() - t0

    results["data"]["obj"].append(hist_data["obj"])
    results["data"]["pdg"].append(hist_data["pdg"])
    results["data"]["init_time_s"].append(t_init)
    results["data"]["solve_time_s"].append(t_solve)
    results["data"]["total_time_s"].append(t_init + t_solve)

    # random init
    t0 = time.perf_counter()
    U0_rand, P0_rand = random_init(X, m, lam, w, seed + t)
    t_init = time.perf_counter() - t0

    t0 = time.perf_counter()
    _, _, hist_rand = run_vanilla_pdhg(X, src, dst, w, lam, tau, sigma, iters, U0_rand, P0_rand)
    t_solve = time.perf_counter() - t0

    results["random"]["obj"].append(hist_rand["obj"])
    results["random"]["pdg"].append(hist_rand["pdg"])
    results["random"]["init_time_s"].append(t_init)
    results["random"]["solve_time_s"].append(t_solve)
    results["random"]["total_time_s"].append(t_init + t_solve)

    # pdhg-net init
    t0 = time.perf_counter()
    U0_net, P0_net = pdhg_net_init(model, g, device)
    t_init = time.perf_counter() - t0

    t0 = time.perf_counter()
    _, _, hist_net = run_vanilla_pdhg(X, src, dst, w, lam, tau, sigma, iters, U0_net, P0_net)
    t_solve = time.perf_counter() - t0

    results["pdhg_net"]["obj"].append(hist_net["obj"])
    results["pdhg_net"]["pdg"].append(hist_net["pdg"])
    results["pdhg_net"]["init_time_s"].append(t_init)
    results["pdhg_net"]["solve_time_s"].append(t_solve)
    results["pdhg_net"]["total_time_s"].append(t_init + t_solve)

In [ ]:
def arr(method, key):
    return np.asarray(results[method][key], dtype=np.float64)


def first_hit(curve, target):
    idx = np.where(curve <= target)[0]
    return float(idx[0]) if len(idx) > 0 else np.nan


# iterations to threshold objective
iter_summary = {m: [] for m in methods}
for t in range(n_trials):
    finals = [arr(m, "obj")[t, -1] for m in methods]
    best_final = min(finals)
    target = best_final * (1.0 + objective_rel_tol)
    for m in methods:
        iter_summary[m].append(first_hit(arr(m, "obj")[t], target))

print(f"Trials: {n_trials}, n_samples={n_samples}, noise={noise}, k={k_neighbors}")
print(f"Target objective: (1 + {objective_rel_tol:.2%}) * best final objective")
for m in methods:
    it = np.asarray(iter_summary[m], dtype=np.float64)
    print(f"{m:9s} | mean_iter={np.nanmean(it):8.2f} | median_iter={np.nanmedian(it):8.2f} | hit_rate={np.mean(~np.isnan(it)):6.2%}")

print("\nRuntime per trial (seconds, mean +/- std):")
for m in methods:
    ti = arr(m, "init_time_s")
    ts = arr(m, "solve_time_s")
    tt = arr(m, "total_time_s")
    print(f"{m:9s} | init={ti.mean():7.4f} +/- {ti.std():7.4f} | solve={ts.mean():7.4f} +/- {ts.std():7.4f} | total={tt.mean():7.4f} +/- {tt.std():7.4f}")


In [ ]:
# mean curves with error bars across trials
colors = {"data": "tab:blue", "random": "tab:orange", "pdhg_net": "tab:green"}
labels = {"data": "data init", "random": "random init", "pdhg_net": "pdhg-net init"}

fig, axs = plt.subplots(1, 2, figsize=(13, 4))
for metric, ax in zip(["obj", "pdg"], axs):
    for m in methods:
        A = arr(m, metric)
        mu = A.mean(axis=0)
        sd = A.std(axis=0)
        x = np.arange(A.shape[1])

        # mean curve
        ax.plot(x, mu, color=colors[m], label=labels[m])

        # error bars every error_every iter
        xe = x[::error_every]
        mue = mu[::error_every]
        sde = sd[::error_every]
        ax.errorbar(xe, mue, yerr=sde, fmt="none", ecolor=colors[m], alpha=0.35, capsize=2)

    ax.set_title(metric)
    ax.set_xlabel("PDHG iteration")
    ax.set_yscale("log")
    ax.grid(alpha=0.25)

axs[0].legend(loc="best")
plt.tight_layout()
plt.show()

In [ ]:
# runtime bars
stat_keys = ["init_time_s", "solve_time_s", "total_time_s"]
stat_names = ["init", "solve", "total"]

x = np.arange(len(methods))
width = 0.24

fig, ax = plt.subplots(figsize=(8, 4))
for i, (k, name) in enumerate(zip(stat_keys, stat_names)):
    vals = [arr(m, k).mean() for m in methods]
    errs = [arr(m, k).std() for m in methods]
    ax.bar(x + (i - 1) * width, vals, yerr=errs, capsize=3, width=width, label=name)

ax.set_xticks(x)
ax.set_xticklabels(methods)
ax.set_ylabel("seconds")
ax.set_title("Runtime across 2-moons trials (mean +/- std)")
ax.grid(axis="y", alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()
